In [68]:
import pandas as pd

In [69]:
df = pd.read_csv('data.csv')
df.head()

,price,rooms,total_area_m2,floor,address,title,url
0,16 000 000 ₽427 807 ₽/м²,1,37.4 м²,6/7,Москва,"1-комн. квартира, 37,4 м², 6/7 этаж",https://www.cian.ru/cat.php?id_user=17950754&d...
1,25 000 000 ₽438 596 ₽/м²,2,57 м²,4/5,Москва,"2-комн. квартира, 57 м², 4/5 этаж",https://www.cian.ru/cat.php?id_user=131486046&...
2,19 900 000 ₽209 034 ₽/м²,4,95.2 м²,1/5,Москва,"4-комн. квартира, 95,2 м², 1/5 этаж",https://www.cian.ru/cat.php?id_user=73781772&d...
3,9 150 000 ₽150 000 ₽/м²,2,61 м²,2/3,Москва,"2-комн. квартира, 61 м², 2/3 этаж",https://www.cian.ru/cat.php?id_user=9679252&de...
4,22 500 000 ₽703 125 ₽/м²,1,32 м²,7/13,Москва,"1-комн. апартаменты, 32 м², 7/13 этаж",https://www.cian.ru/cat.php?id_user=12789323&d...


In [70]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 361 entries, 0 to 360
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   price          361 non-null    str  
 1   rooms          361 non-null    str  
 2   total_area_m2  361 non-null    str  
 3   floor          361 non-null    str  
 4   address        361 non-null    str  
 5   title          361 non-null    str  
 6   url            361 non-null    str  
dtypes: str(7)
memory usage: 19.9 KB


In [71]:
df.price = df.price.str.split().str[:3].str.join(' ')

In [72]:
df.price

0      16 000 000
1      25 000 000
2      19 900 000
3       9 150 000
4      22 500 000
          ...    
356    30 000 000
357    29 990 000
358    29 900 000
359    30 000 000
360    30 000 000
Name: price, Length: 361, dtype: object

In [73]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 361 entries, 0 to 360
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   price          361 non-null    object
 1   rooms          361 non-null    str   
 2   total_area_m2  361 non-null    str   
 3   floor          361 non-null    str   
 4   address        361 non-null    str   
 5   title          361 non-null    str   
 6   url            361 non-null    str   
dtypes: object(1), str(6)
memory usage: 19.9+ KB


In [74]:
df.drop(columns=['address'], inplace=True)
df.head()

,price,rooms,total_area_m2,floor,title,url
0,16 000 000,1,37.4 м²,6/7,"1-комн. квартира, 37,4 м², 6/7 этаж",https://www.cian.ru/cat.php?id_user=17950754&d...
1,25 000 000,2,57 м²,4/5,"2-комн. квартира, 57 м², 4/5 этаж",https://www.cian.ru/cat.php?id_user=131486046&...
2,19 900 000,4,95.2 м²,1/5,"4-комн. квартира, 95,2 м², 1/5 этаж",https://www.cian.ru/cat.php?id_user=73781772&d...
3,9 150 000,2,61 м²,2/3,"2-комн. квартира, 61 м², 2/3 этаж",https://www.cian.ru/cat.php?id_user=9679252&de...
4,22 500 000,1,32 м²,7/13,"1-комн. апартаменты, 32 м², 7/13 этаж",https://www.cian.ru/cat.php?id_user=12789323&d...


In [75]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from qdrant_client import QdrantClient
from dotenv import load_dotenv
import os

In [76]:
load_dotenv()
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

client = QdrantClient(
    url=QDRANT_URL, 
    api_key=QDRANT_API_KEY
)

In [77]:
client.create_collection(
    collection_name="cian",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
)

True

In [80]:
from langchain_huggingface import HuggingFaceEmbeddings

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model_kwargs = {'device': 'mps'}
encode_kwargs = {'normalize_embeddings': True, 'batch_size':128}

embeddings_model = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs,
)

/Users/dmitrijpavlov/ds_git/FastAPI/backend/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2466.87it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [81]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 361 entries, 0 to 360
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   price          361 non-null    object
 1   rooms          361 non-null    str   
 2   total_area_m2  361 non-null    str   
 3   floor          361 non-null    str   
 4   title          361 non-null    str   
 5   url            361 non-null    str   
dtypes: object(1), str(5)
memory usage: 17.1+ KB


In [82]:
from uuid import uuid4
from langchain_core.documents import Document

# Создаем документы для Qdrant с UUID
documents = []
id_mapping = {}  # Сохраним соответствие UUID -> original_id

for _, row in df.iterrows():

    content = row['title']


    # Метаданные
    metadata = {
        "price": row.get('price', ''),
        "rooms": row.get('rooms', ''),
        "total_area_m2": row.get('total_area_m2', ''),
        "floor": row.get('floor', ''),
        "url": row.get('url', ''),     
    }

    documents.append(Document(page_content=content, metadata=metadata))


uuids = [str(uuid4()) for _ in range(len(documents))]

print(f"✅ Создано {len(documents)} документов с UUID")

✅ Создано 361 документов с UUID


In [83]:
documents[0]

Document(metadata={'price': '16 000 000', 'rooms': '1', 'total_area_m2': '37.4 м²', 'floor': '6/7', 'url': 'https://www.cian.ru/cat.php?id_user=17950754&deal_type=sale&flat=yes&engine_version=2'}, page_content='1-комн. квартира, 37,4 м², 6/7 этаж')

In [84]:
from langchain_qdrant import QdrantVectorStore

vector_store = QdrantVectorStore(
    client=client,
    collection_name="cian",
    embedding=embeddings_model
)

In [85]:
from tqdm import tqdm

batch_size = 4
total_batches = (len(documents) + batch_size - 1) // batch_size


with tqdm(total=total_batches, desc="Добавление батчей в Qdrant") as pbar:
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i+batch_size]
        batch_ids = uuids[i:i+batch_size]

        vector_store.add_documents(documents=batch_docs, ids=batch_ids)
        pbar.update(1)

print(f"✅ {len(documents)} документов добавлено в Qdrant!")

Добавление батчей в Qdrant: 100%|██████████| 91/91 [00:10<00:00,  8.33it/s]

✅ 361 документов добавлено в Qdrant!


In [86]:
points, _ = client.scroll(
    collection_name="cian",
    limit=2,
    with_payload=True,
    with_vectors=True
)
points[1].payload['metadata']

{'price': '14 399 000',
 'rooms': '1',
 'total_area_m2': '35 м²',
 'floor': '3/12',
 'url': 'https://www.cian.ru/cat.php?id_user=15561797&deal_type=sale&flat=yes&engine_version=2'}